## Fast-Run Sanity Check (Single Model)

This notebook is a lightweight variant of full training:
- single model: `efficientnet_b2`
- fewer epochs (quick check)
- same leakage guard (`__aug_` excluded from val/test)
- basic threshold tuning + test evaluation

Use this to validate data pipeline and training setup quickly before full runs.

In [ ]:
!pip install -q timm

In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

from torchvision import transforms
import timm

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score


@dataclass
class CFG:
    train_dir: str = '/kaggle/input/datasets/edamhakim/tongue-preped/train'
    val_dir: str = '/kaggle/input/datasets/edamhakim/tongue-preped/val'
    test_dir: str = '/kaggle/input/datasets/edamhakim/tongue-preped/test'
    output_dir: str = '/kaggle/working'

    img_size: int = 224
    batch_size: int = 32
    num_workers: int = 2
    pin_memory: bool = True

    warmup_epochs: int = 2
    finetune_epochs: int = 4
    head_lr: float = 1e-3
    ft_lr: float = 1e-4
    eta_min: float = 1e-6
    weight_decay: float = 1e-4
    early_stopping_patience: int = 3

    mixup_alpha: float = 0.2
    seed: int = 42


cfg = CFG()
class_to_label = {'diabetes': 1, 'nondiabetes': 0}


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
class TongueDataset(Dataset):
    def __init__(self, root_dir: str, class_to_label: Dict[str, int], transform=None, split_name='train'):
        self.samples: List[Tuple[Path, int]] = []
        self.transform = transform
        root = Path(root_dir)

        for class_name, label in class_to_label.items():
            cdir = root / class_name
            if not cdir.exists():
                continue
            for p in sorted(cdir.glob('*.jpg')):
                if split_name in ('val', 'test') and '__aug_' in p.name.lower():
                    continue
                self.samples.append((p, label))

        if len(self.samples) == 0:
            raise RuntimeError(f'No samples in {root_dir}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p, y = self.samples[idx]
        x = Image.open(p).convert('RGB')
        if self.transform:
            x = self.transform(x)
        return x, torch.tensor(y, dtype=torch.float32), str(p)


train_tfms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = TongueDataset(cfg.train_dir, class_to_label, train_tfms, 'train')
val_ds = TongueDataset(cfg.val_dir, class_to_label, val_test_tfms, 'val')
test_ds = TongueDataset(cfg.test_dir, class_to_label, val_test_tfms, 'test')


def print_stats(name, ds):
    ys = [y for _, y in ds.samples]
    print(f"{name}: total={len(ds)} | nondiabetes={ys.count(0)} | diabetes={ys.count(1)}")


print_stats('train (orig+aug)', train_ds)
print_stats('val (orig only)', val_ds)
print_stats('test (orig only)', test_ds)

train_labels = [y for _, y in train_ds.samples]
class_count = np.bincount(train_labels)
class_weights = 1.0 / np.maximum(class_count, 1)
sample_weights = [class_weights[y] for y in train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

In [ ]:
def build_model():
    model = timm.create_model('efficientnet_b2', pretrained=True)
    model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1408, 1))
    return model


def freeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True


def partial_unfreeze(model):
    freeze_backbone(model)
    for p in model.blocks[-2:].parameters():
        p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True


def trainable_params(model):
    return [p for p in model.parameters() if p.requires_grad]


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam


criterion = nn.BCEWithLogitsLoss()


def evaluate(model, loader):
    model.eval()
    losses, probs, labels, paths = [], [], [], []
    with torch.no_grad():
        for x, y, p in loader:
            x, y = x.to(device), y.to(device)
            with autocast(enabled=(device.type == 'cuda')):
                logits = model(x).squeeze(1)
                loss = criterion(logits, y)
            losses.append(loss.item())
            probs.extend(torch.sigmoid(logits).cpu().numpy().tolist())
            labels.extend(y.cpu().numpy().tolist())
            paths.extend(p)
    labels = np.array(labels)
    probs = np.array(probs)
    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.5
    return {'loss': float(np.mean(losses)), 'auc': float(auc), 'labels': labels, 'probs': probs, 'paths': paths}


model = build_model().to(device)
scaler = GradScaler(enabled=(device.type == 'cuda'))

freeze_backbone(model)
optimizer = optim.AdamW(trainable_params(model), lr=cfg.head_lr, weight_decay=cfg.weight_decay)
scheduler = None

best_auc = -1.0
best_path = Path(cfg.output_dir) / 'efficientnet_b2_fast_best.pt'
no_improve = 0
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

total_epochs = cfg.warmup_epochs + cfg.finetune_epochs
for epoch in range(1, total_epochs + 1):
    if epoch == cfg.warmup_epochs + 1:
        partial_unfreeze(model)
        optimizer = optim.AdamW(trainable_params(model), lr=cfg.ft_lr, weight_decay=cfg.weight_decay)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.finetune_epochs, eta_min=cfg.eta_min)

    use_mixup = epoch > cfg.warmup_epochs
    model.train()
    train_losses = []

    for x, y, _ in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=(device.type == 'cuda')):
            if use_mixup:
                xm, ya, yb, lam = mixup_data(x, y, alpha=cfg.mixup_alpha)
                logits = model(xm).squeeze(1)
                loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)
            else:
                logits = model(x).squeeze(1)
                loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())

    if scheduler is not None:
        scheduler.step()

    val_out = evaluate(model, val_loader)
    train_loss = float(np.mean(train_losses))

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_out['loss'])
    history['val_auc'].append(val_out['auc'])

    print(f"Epoch {epoch:02d}/{total_epochs} | train_loss={train_loss:.4f} | val_loss={val_out['loss']:.4f} | val_AUC={val_out['auc']:.4f}")

    if val_out['auc'] > best_auc:
        best_auc = val_out['auc']
        no_improve = 0
        torch.save(model.state_dict(), best_path)
    else:
        no_improve += 1

    if no_improve >= cfg.early_stopping_patience:
        print('Early stopping triggered.')
        break

print('Best checkpoint:', best_path)
print('Best val AUC:', best_auc)

In [ ]:
# Load best checkpoint and tune threshold on validation
best_model = build_model().to(device)
best_model.load_state_dict(torch.load(best_path, map_location=device))
best_model.eval()

val_out = evaluate(best_model, val_loader)
thresholds = np.round(np.arange(0.30, 0.701, 0.01), 2)
threshold_scores = []
for t in thresholds:
    pred = (val_out['probs'] >= t).astype(int)
    macro_f1 = f1_score(val_out['labels'], pred, average='macro', zero_division=0)
    threshold_scores.append((t, macro_f1))

best_threshold, best_macro_f1 = max(threshold_scores, key=lambda x: x[1])
print(f'Best threshold: {best_threshold:.2f} (val macro F1={best_macro_f1:.4f})')

# Final test evaluation
test_out = evaluate(best_model, test_loader)
test_pred = (test_out['probs'] >= best_threshold).astype(int)

test_metrics = {
    'AUC-ROC': float(roc_auc_score(test_out['labels'], test_out['probs']) if len(np.unique(test_out['labels'])) > 1 else 0.5),
    'Accuracy': float(accuracy_score(test_out['labels'], test_pred)),
    'F1': float(f1_score(test_out['labels'], test_pred, zero_division=0)),
    'Precision': float(precision_score(test_out['labels'], test_pred, zero_division=0)),
    'Recall': float(recall_score(test_out['labels'], test_pred, zero_division=0)),
}

print('\nFast-run test metrics:')
print(pd.DataFrame([test_metrics]).to_string(index=False))

# Export quick submission-style file
fast_submission = pd.DataFrame({
    'filepath': test_out['paths'],
    'true_label': test_out['labels'].astype(int),
    'predicted_prob': test_out['probs'],
})
fast_submission['predicted_label'] = (fast_submission['predicted_prob'] >= best_threshold).astype(int)
fast_submission_path = Path(cfg.output_dir) / 'submission_fast.csv'
fast_submission.to_csv(fast_submission_path, index=False)
print('\nSaved:', fast_submission_path)